# FLIR flash-method thermal analysis in Google Colab

This notebook runs the standalone thermal-analysis workflow from GitHub. It installs the required packages, reads FLIR videos from Google Drive or `/content`, runs the manifest workflow, and displays the combined summary table.


In [ ]:
# 1) Clone or update the repository.
REPO_URL = 'https://github.com/AteeqUrRehmanDaudzai/thermal-flash-analysis.git'
REPO_DIR = '/content/thermal-flash-analysis'

import os
import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('Working in', Path.cwd())


In [ ]:
# 2) Install analysis dependencies.
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'thermal_flash_analysis/requirements.txt'], check=True)


In [ ]:
# 3) Mount Google Drive if your videos are stored there.
# Skip this cell if you uploaded videos directly to /content.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 4) Configure your analysis.
from pathlib import Path

VIDEO_DIR = Path('/content/drive/MyDrive/FLIR_flash/data/raw')
OUTPUT_DIR = Path('/content/drive/MyDrive/FLIR_flash/results')
MANIFEST = Path('thermal_flash_analysis/experiment_manifest.csv')

SAMPLE_THICKNESS_M = 0.0011  # Replace with measured total package thickness in metres.
ROI = ''  # Optional format: 'x,y,width,height'. Leave blank for full frame.
MAKE_VIDEO = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Video directory:', VIDEO_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
# 5) Run all videos listed in the manifest.
import subprocess
import sys

cmd = [
    sys.executable, 'thermal_flash_analysis/analyze_flash.py',
    '--manifest', str(MANIFEST),
    '--video-dir', str(VIDEO_DIR),
    '--sample-thickness-m', str(SAMPLE_THICKNESS_M),
    '--output-dir', str(OUTPUT_DIR),
]
if ROI:
    cmd.extend(['--roi', ROI])
if MAKE_VIDEO:
    cmd.append('--make-video')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 6) View the combined summary table.
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / 'all_summaries.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
    display(summary.groupby(['sample', 'configuration'])['effective_diffusivity_m2_s'].agg(['count', 'mean', 'std']))
else:
    print('No all_summaries.csv found. Check VIDEO_DIR and manifest file names.')


In [ ]:
# 7) Zip results for download if desired.
import shutil
zip_base = '/content/flir_flash_results'
archive = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
print('Created', archive)
